# Latent Diffusion on Anime Faces  <sup>🧨 `diffusers`</sup>

A **latent diffusion model (LDM)** for unconditional anime-face generation — Stable Diffusion's family, minus the text conditioning, stripped down to the essentials.

**The one-line idea:** pixels are expensive to diffuse directly, so we compress each image into a small *latent* with a **pretrained, frozen autoencoder**, run the entire diffusion process in that tiny latent space, then decode the result back to an image.

**What's pretrained vs. what we train**

| Component | Role | Trained here? |
|---|---|---|
| **VQ autoencoder** (`vq-f4`) | pixels ⇄ latents | ❌ frozen, pretrained |
| **UNet** (`UNet2DModel`) | denoises *in latent space* | ✅ the only thing we train |
| **Scheduler** (DDPM / DDIM) | noise schedule + sampling | — (no params) |

**Why a VQ autoencoder and not SDXL's KL-VAE?** Those are 8× downsamplers tuned for large, photographic, text-conditioned generation — overkill for 64px anime faces, slow, and in practice they reconstruct these simple faces poorly at this scale. We instead use **`vq-f4`** from the original LDM paper (`CompVis/ldm-celebahq-256`):

- **only 4× downsampling** → keeps fine detail (64×64 → **16×16** latents),
- **trained on faces** (CelebA-HQ) → a near-perfect prior for anime faces,
- **just 3 latent channels** → a small, fast diffusion UNet,
- **loads natively in `diffusers`**, no gated licenses.

The "VQ" (vector-quantizer) is the key trick: we run diffusion on the *continuous* latents, and the frozen decoder's **pretrained codebook snaps** each generated latent to clean, valid codes before decoding — so even a slightly-off diffusion sample decodes to a crisp face.

## Setup

In [ ]:
#@title install
from IPython.display import clear_output
!pip install -q diffusers accelerate torchinfo kagglehub
clear_output()
print("installed ✓")

In [ ]:
#@title imports
import os, math
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Lambda, RandomHorizontalFlip
import torchvision.utils as vutils

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from diffusers import VQModel, UNet2DModel, DDPMScheduler, DDIMScheduler

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## Data

Anime faces, resized to **64×64** and normalised to **[-1, 1]** (what the autoencoder expects). The frozen `vq-f4` encoder downsamples 4×, turning each image into a **16×16×3** latent — a comfortable resolution for the diffusion UNet.

In [ ]:
#@title download dataset
import kagglehub
path = kagglehub.dataset_download("soumikrakshit/anime-faces")
dataset_path = os.path.join(path, "data")
clear_output()
print("dataset:", dataset_path)

In [ ]:
#@title dataset + dataloaders
IMAGE_SIZE   = 64
BATCH_SIZE   = 64
VAL_FRACTION = 0.05

transform = Compose([
    Resize(IMAGE_SIZE),
    CenterCrop(IMAGE_SIZE),
    RandomHorizontalFlip(p=0.5),      # free augmentation for faces
    ToTensor(),                       # -> [0, 1]
    Lambda(lambda t: t * 2 - 1),      # -> [-1, 1]
])

class AnimeFaces(Dataset):
    def __init__(self, root, transform):
        self.root, self.transform = root, transform
        self.files = [f for f in os.listdir(root)
                      if f.lower().endswith((".png", ".jpg", ".jpeg"))]
    def __len__(self):  return len(self.files)
    def __getitem__(self, i):
        img = Image.open(os.path.join(self.root, self.files[i])).convert("RGB")
        return self.transform(img), 0

full = AnimeFaces(dataset_path, transform)
val_n   = int(len(full) * VAL_FRACTION)
train_n = len(full) - val_n
g = torch.Generator().manual_seed(42)
train_set, val_set = torch.utils.data.random_split(full, [train_n, val_n], generator=g)

workers = min(4, os.cpu_count() or 2)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=workers, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=workers, pin_memory=True)

print(f"train {train_n:,} | val {val_n:,} | batches/epoch {len(train_loader)}")

In [ ]:
#@title peek at the data
imgs = torch.stack([train_set[i][0] for i in range(8)])
grid = vutils.make_grid((imgs + 1) / 2, nrow=8, padding=2)
plt.figure(figsize=(14, 2)); plt.axis("off")
plt.imshow(grid.permute(1, 2, 0).numpy()); plt.title("training samples"); plt.show()

## The frozen VQ autoencoder (`vq-f4`)

We load the pretrained **`vq-f4`** autoencoder from `CompVis/ldm-celebahq-256` and freeze it. It maps a **64×64×3** image to a **16×16×3** latent and back.

Two things worth understanding about `VQModel`:

- **`encode(x).latents`** returns the *continuous* latent (before quantization). This is what we run diffusion on — it's smooth and well-behaved for the denoising objective.
- **`decode(z).sample`** *internally applies the pretrained vector-quantizer* — it snaps `z` to the nearest codebook entries, then decodes. This is the "VQ" step that cleans up generated latents into crisp faces.

We also compute a single **`LATENT_SCALE`** from one batch so the latents fed to the UNet have ≈ unit standard deviation (diffusion assumes roughly unit-variance data). We divide by it on encode and multiply back on decode.

In [ ]:
#@title login
from getpass import getpass

# Prompts you to paste the token — it won't be echoed or saved in notebook output
HF_TOKEN = getpass("Enter your Hugging Face token: ")
from huggingface_hub import login

login(token=HF_TOKEN)
import os
print(f'logged in')

In [ ]:
#@title load + freeze the VQ autoencoder
vae = VQModel.from_pretrained("CompVis/ldm-celebahq-256", subfolder="vqvae").to(device)
vae.eval().requires_grad_(False)

LATENT_CHANNELS = vae.config.latent_channels                      # 3
DOWN_FACTOR     = 2 ** (len(vae.config.block_out_channels) - 1)   # 4
LATENT_SIZE     = IMAGE_SIZE // DOWN_FACTOR                        # 16
print(f"vq-f4 loaded — latent = {LATENT_CHANNELS}×{LATENT_SIZE}×{LATENT_SIZE} ({DOWN_FACTOR}× downsample)")

In [ ]:
#@title estimate the latent scaling factor (one batch)
batch0, _ = next(iter(train_loader))
with torch.no_grad():
    h0 = vae.encode(batch0.to(device)).latents
LATENT_SCALE = h0.std().item()
print(f"LATENT_SCALE = {LATENT_SCALE:.4f}   (raw latent std)")

@torch.no_grad()
def encode(images):
    # [-1,1] pixels -> unit-scaled continuous latents
    return vae.encode(images.to(device)).latents / LATENT_SCALE

@torch.no_grad()
def decode(latents):
    # unit-scaled latents -> [-1,1] pixels (quantizer applied inside decode)
    return vae.decode(latents * LATENT_SCALE).sample

In [ ]:
#@title sanity check: encode -> decode a real batch
batch, _ = next(iter(val_loader))
batch = batch[:6].to(device)
with torch.no_grad():
    recon = decode(encode(batch))

def denorm(x): return (x.clamp(-1, 1) + 1) / 2
rows = []
for i in range(batch.shape[0]):
    rows += [denorm(batch[i]).cpu(), denorm(recon[i]).cpu()]
grid = vutils.make_grid(rows, nrow=2, padding=2)
plt.figure(figsize=(4, 11)); plt.axis("off")
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title("original  |  reconstruction\n(this is the quality ceiling)"); plt.show()

## The diffusion UNet

The **only trained component**. A standard `UNet2DModel` operating on **16×16×3** latents instead of pixels. Attention lives at the two coarser resolutions (8×8 and 4×4) where it coordinates global face structure cheaply; the top 16×16 level stays convolutional.

In [ ]:
#@title latent-space UNet
model = UNet2DModel(
    sample_size=LATENT_SIZE,                        # 16
    in_channels=LATENT_CHANNELS,                    # 3
    out_channels=LATENT_CHANNELS,                   # 3
    layers_per_block=2,
    block_out_channels=(128, 256, 384),             # 16 -> 8 -> 4
    down_block_types=("DownBlock2D",                # 16
                      "AttnDownBlock2D",            # 8   (attention)
                      "AttnDownBlock2D"),           # 4   (attention)
    up_block_types  =("AttnUpBlock2D",              # 4
                      "AttnUpBlock2D",              # 8
                      "UpBlock2D"),                 # 16
    norm_num_groups=32,                             # divides every channel count
    act_fn="silu",
).to(device)

print(f"UNet params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## Noise schedulers

A 1000-step cosine **DDPM** schedule for training (add noise + sample timesteps), and a fast **DDIM** schedule for sampling. Identical math to a pixel-space DDPM — it just runs on latents.

In [ ]:
#@title schedulers
noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",   # cosine — good default for faces
)

sampling_scheduler = DDIMScheduler.from_config(noise_scheduler.config)
sampling_scheduler.set_timesteps(50)

print("train steps:", noise_scheduler.config.num_train_timesteps,
      "| DDIM sample steps:", len(sampling_scheduler.timesteps))

## Training objective

Plain **ε-prediction** on latents: encode the image, add noise to the latent at a random timestep, and have the UNet predict the noise. Nothing fancy — a single MSE.

In [ ]:
#@title diffusion loss (Min-SNR-γ weighted)
SNR_GAMMA = 5.0   # standard value from the Min-SNR paper

def diffusion_loss(images):
    latents = encode(images)                                     # frozen VAE, no grad
    noise   = torch.randn_like(latents)
    t = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                      (latents.shape[0],), device=device).long()
    noisy   = noise_scheduler.add_noise(latents, noise, t)
    pred    = model(noisy, t).sample

    # per-sample MSE (mean over channel + spatial dims, keep the batch dim)
    mse = F.mse_loss(pred, noise, reduction="none").mean(dim=[1, 2, 3])

    # SNR(t) = ᾱ_t / (1 - ᾱ_t)
    acp = noise_scheduler.alphas_cumprod.to(device)[t]
    snr = acp / (1 - acp)

    # ε-prediction weight: min(SNR, γ) / SNR
    weight = torch.clamp(snr, max=SNR_GAMMA) / snr

    return (weight * mse).mean()

## Sampling

Start from Gaussian noise **in latent space**, run the DDIM reverse loop, then **decode** to pixels (the frozen quantizer + decoder do the cleanup).

In [ ]:
#@title sampler + grid helper
@torch.no_grad()
def sample(n=16, scheduler=sampling_scheduler):
    was_training = model.training
    model.eval()
    lat = torch.randn(n, LATENT_CHANNELS, LATENT_SIZE, LATENT_SIZE, device=device)
    for t in scheduler.timesteps:
        pred = model(lat, t).sample
        lat  = scheduler.step(pred, t, lat).prev_sample
    imgs = (decode(lat).clamp(-1, 1) + 1) / 2
    if was_training: model.train()
    return imgs.cpu()

def show_grid(imgs, title="", cols=8):
    n = imgs.shape[0]; rows = math.ceil(n / cols)
    plt.figure(figsize=(1.6 * cols, 1.6 * rows))
    grid = vutils.make_grid(imgs, nrow=cols, padding=2)
    plt.imshow(grid.permute(1, 2, 0).numpy()); plt.axis("off")
    if title: plt.title(title)
    plt.show()

## Training

A compact step-based loop with **mixed precision**. Every so often it:
- **logs** the running train loss,
- **evaluates** the validation loss,
- **previews** generated samples so you can *watch* faces emerge (diffusion loss plateaus early and is a poor quality signal — the previews are what matter),
- **flushes** the matplotlib output so the notebook stays light.

> The cell is built to **run correctly end-to-end**; set `TOTAL_STEPS` to something real (e.g. 8000–15000) when you actually want quality.

In [ ]:
#@title training config
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

TOTAL_STEPS   = 40000     # bump to 8k–15k for good results
LR            = 2e-4
GRAD_CLIP     = 1.0

LOG_EVERY     = 50       # record train loss
VAL_EVERY     = 250      # record val loss
PREVIEW_EVERY = 500      # generate + show samples
FLUSH_EVERY   = 500      # clear cell output

optimizer    = AdamW(model.parameters(), lr=LR)
lr_scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_STEPS)
scaler       = torch.amp.GradScaler(device)

train_hist, val_hist = [], []   # each: (step, loss)
print(f"configured for {TOTAL_STEPS:,} steps")

In [ ]:
#@title validation loss
@torch.no_grad()
def validate():
    model.eval()
    tot = n = 0
    for images, _ in val_loader:
        with torch.autocast(device, enabled=(device == "cuda")):
            tot += diffusion_loss(images).item(); n += 1
    model.train()
    return tot / n

In [ ]:
#@title train
from IPython.display import clear_output

def cycle(loader):
    while True:
        for b in loader: yield b
train_iter = cycle(train_loader)

model.train()
running = 0.0
pbar = tqdm(range(1, TOTAL_STEPS + 1), desc="training")

for step in pbar:
    images, _ = next(train_iter)

    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device, enabled=(device == "cuda")):
        loss = diffusion_loss(images)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer); scaler.update()
    lr_scheduler.step()

    running += loss.item()

    if step % LOG_EVERY == 0:
        avg = running / LOG_EVERY; running = 0.0
        train_hist.append((step, avg))
        pbar.set_postfix(loss=f"{avg:.4f}", lr=f"{lr_scheduler.get_last_lr()[0]:.1e}")

    if step % VAL_EVERY == 0:
        val_hist.append((step, validate()))

    if step % FLUSH_EVERY == 0:
        clear_output(wait=True)

    if step % PREVIEW_EVERY == 0:
        show_grid(sample(n=8), title=f"samples @ step {step}", cols=8)

print("training complete ✓")

## Training history

In [ ]:
#@title loss curves
plt.figure(figsize=(9, 4))
if train_hist:
    ts, ls = zip(*train_hist); plt.plot(ts, ls, label="train", lw=1.5)
if val_hist:
    vs, vl = zip(*val_hist);   plt.plot(vs, vl, label="val", lw=1.5, marker="o", ms=4)
plt.xlabel("step"); plt.ylabel("MSE (ε-prediction)")
plt.title("latent diffusion training"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Generation

In [ ]:
#@title generate a grid of faces
show_grid(sample(n=16), title="generated anime faces (50-step DDIM)", cols=8)

In [ ]:
#@title higher-quality — more DDIM steps
steps = 200
hq = DDIMScheduler.from_config(noise_scheduler.config)
hq.set_timesteps(steps)
show_grid(sample(n=16, scheduler=hq), title=f"generated anime faces ({steps}-step DDIM)", cols=8)

In [ ]:
#@title upload trained model to Hugging Face
from huggingface_hub import HfApi, create_repo, whoami, login
import torch, json

# login()   # <- uncomment on first run, paste a WRITE token

REPO_NAME = "anime_LDM_CompVis_VQVAE_40k_steps"
username  = whoami()["name"]
repo_id   = f"{username}/{REPO_NAME}"
print("repo:", repo_id)

# 1. UNet weights + its config (the only trained component)
model.save_pretrained("anime_LDM_upload")

# 2. scheduler config (so sampling is reproducible)
noise_scheduler.save_config("anime_LDM_upload/scheduler")

# 3. everything needed to resume training
torch.save({
    "optimizer":    optimizer.state_dict(),
    "lr_scheduler": lr_scheduler.state_dict(),
    "scaler":       scaler.state_dict(),
    "step":         step,                  # so you know where to resume from
    "train_hist":   train_hist,
    "val_hist":     val_hist,
}, "anime_LDM_upload/training_state.pt")

# 4. metadata: the constants sampling depends on
meta = {
    "latent_scale":    LATENT_SCALE,       # NOT recoverable without this
    "latent_channels": LATENT_CHANNELS,
    "latent_size":     LATENT_SIZE,
    "image_size":      IMAGE_SIZE,
    "vqvae_repo":      "CompVis/ldm-celebahq-256",               # your own VQVAE repo
    "vqvae_ckpt":      "vqvae",
    "snr_gamma":       SNR_GAMMA,
    "total_steps":     TOTAL_STEPS,
}
with open("anime_LDM_upload/meta.json", "w") as f:
    json.dump(meta, f, indent=2)

# 5. push the folder
create_repo(repo_id, exist_ok=True)
HfApi().upload_folder(
    folder_path="anime_LDM_upload",
    repo_id=repo_id,
    commit_message="Anime latent-diffusion UNet + scheduler + training state",
)
print("done →", f"https://huggingface.co/{repo_id}")

In [ ]:
#@title load the trained model from Hugging Face
import torch, json
from huggingface_hub import hf_hub_download, whoami
from diffusers import UNet2DModel, VQModel, DDPMScheduler, DDIMScheduler

REPO_NAME = "anime_LDM_CompVis_VQVAE_40k_steps"
repo_id   = f"{whoami()['name']}/{REPO_NAME}"
device    = "cuda" if torch.cuda.is_available() else "cpu"


def load_meta(repo_id):
    with open(hf_hub_download(repo_id, "meta.json")) as f:
        return json.load(f)


def load_model(repo_id, device):
    return UNet2DModel.from_pretrained(repo_id).to(device)


def load_schedulers(repo_id, sample_steps=50):
    noise_sched = DDPMScheduler.from_pretrained(repo_id, subfolder="scheduler")
    ddim = DDIMScheduler.from_config(noise_sched.config)
    ddim.set_timesteps(sample_steps)
    return noise_sched, ddim


def load_vae(meta, device):
    v = VQModel.from_pretrained(meta["vae_repo"], subfolder=meta["vae_subfolder"])
    return v.to(device).eval().requires_grad_(False)


def load_training_state(repo_id):
    path = hf_hub_download(repo_id, "training_state.pt")
    return torch.load(path, map_location="cpu")


# --- load everything ---
meta         = load_meta(repo_id)
model        = load_model(repo_id, device)
vae          = load_vae(meta, device)
noise_scheduler, sampling_scheduler = load_schedulers(repo_id, sample_steps=50)
state        = load_training_state(repo_id)

# restore the constants sampling needs
LATENT_SCALE    = meta["latent_scale"]
LATENT_CHANNELS = meta["latent_channels"]
LATENT_SIZE     = meta["latent_size"]
IMAGE_SIZE      = meta["image_size"]
SNR_GAMMA       = meta["snr_gamma"]
train_hist, val_hist = state["train_hist"], state["val_hist"]
start_step      = state.get("step", 0)

# --- optimizer / lr_scheduler / scaler (for resuming training) ---
optimizer    = torch.optim.AdamW(model.parameters(), lr=1e-4)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=meta["total_steps"])          # match your training constructor
scaler       = torch.amp.GradScaler(enabled=(device == "cuda"))

optimizer.load_state_dict(state["optimizer"])
lr_scheduler.load_state_dict(state["lr_scheduler"])
scaler.load_state_dict(state["scaler"])

model.train()   # switch to .eval() if you only want to sample

print(f"loaded {repo_id}")
print(f"UNet params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(f"latent {LATENT_CHANNELS}×{LATENT_SIZE}×{LATENT_SIZE} | scale {LATENT_SCALE:.4f}")
print(f"history: {len(train_hist)} train / {len(val_hist)} val points | resume @ step {start_step}")
print(f"optimizer lr: {optimizer.param_groups[0]['lr']:.2e}")

## Muhannad's pretrained VQ-VAE

In [ ]:
#@title login
from getpass import getpass

# Prompts you to paste the token — it won't be echoed or saved in notebook output
HF_TOKEN = getpass("Enter your Hugging Face token: ")
from huggingface_hub import login

login(token=HF_TOKEN)
import os
print(f'logged in')

In [ ]:
username  = whoami()["name"]
username

In [ ]:
#@title load + freeze the VQ autoencoder
vae = VQModel.from_pretrained("CompVis/ldm-celebahq-256", subfolder="vqvae").to(device)
vae.eval().requires_grad_(False)

LATENT_CHANNELS = vae.config.latent_channels                      # 3
DOWN_FACTOR     = 2 ** (len(vae.config.block_out_channels) - 1)   # 4
LATENT_SIZE     = IMAGE_SIZE // DOWN_FACTOR                        # 16
print(f"vq-f4 loaded — latent = {LATENT_CHANNELS}×{LATENT_SIZE}×{LATENT_SIZE} ({DOWN_FACTOR}× downsample)")

In [ ]:
#@title model def

import torch
import torch.nn as nn
from huggingface_hub import PyTorchModelHubMixin


class SelfAttention2d(nn.Module):
    def __init__(self, c, heads=4):
        super().__init__()
        self.heads = heads
        self.norm = nn.GroupNorm(min(32, c), c)
        self.qkv  = nn.Conv2d(c, c * 3, 1)
        self.proj = nn.Conv2d(c, c, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        q, k, v = self.qkv(self.norm(x)).chunk(3, dim=1)
        # (B, heads, HW, head_dim)
        q, k, v = [t.view(B, self.heads, C // self.heads, H * W).transpose(-1, -2)
                   for t in (q, k, v)]
        out = F.scaled_dot_product_attention(q, k, v)
        out = out.transpose(-1, -2).reshape(B, C, H, W)
        return x + self.proj(out)                    # residual

def get_gn(channels):
    groups = min(32, channels)
    while channels % groups != 0:
        groups -= 1
    return nn.GroupNorm(groups, channels)


class ResBlock(nn.Module, PyTorchModelHubMixin):
    def __init__(self, c):
        super().__init__()
        self.net = nn.Sequential(
            get_gn(c),
            nn.ReLU(),
            nn.Conv2d(c, c, 3, padding=1),
            get_gn(c),
            nn.ReLU(),
            nn.Conv2d(c, c, 1),
        )

    def forward(self, x):
        return x + self.net(x)


class VQVAE(nn.Module, PyTorchModelHubMixin):
    def __init__(self, dim=64, n_codes=512):
        super().__init__()

        def down(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 4, stride=2, padding=1),
                get_gn(cout),
                nn.ReLU(),
                ResBlock(cout),
                ResBlock(cout),
            )

        def up(cin, cout):
            return nn.Sequential(
                nn.ConvTranspose2d(cin, cout, 4, stride=2, padding=1),
                get_gn(cout),
                nn.ReLU(),
                ResBlock(cout),
                ResBlock(cout),
            )

        # 1. ENCODER
        self.encoder_backbone = nn.Sequential(
            down(3, 64),                                     # 64 -> 32
            down(64, 128),                                   # 32 -> 16
            ResBlock(128),
            SelfAttention2d(128),
            ResBlock(128),
            get_gn(128),
            nn.ReLU(),
        )
        self.to_bottleneck = nn.Conv2d(128, dim, 3, stride=1, padding=1)


        # 2. QUANTIZER (Low-Dim Lookup Setup)
        self.quant_proj = nn.Conv2d(dim, 8, 1)
        self.vq = VectorQuantizer(n_codes, dim=8)  # <--- FIXED: Set to 8 to match quant_proj
        self.post_quant_proj = nn.Conv2d(8, dim, 1)

        # 3. DECODER
        self.from_bottleneck = nn.Conv2d(dim, 128, 3, stride=1, padding=1)

        self.decoder_backbone = nn.Sequential(
            ResBlock(128),
            SelfAttention2d(128),
            ResBlock(128),
            up(128, 64),                                     # 16 -> 32
            nn.ConvTranspose2d(64, 3, 4, stride=2, padding=1), # 32 -> 64
            nn.Tanh(),                                        # <--- DROPPED Sigmoid for Tanh
        )

    def forward(self, x):
        # Encoder
        h = self.encoder_backbone(x)
        z = self.to_bottleneck(h)          # Shape: [B, 64, 16, 16]
        z_low = self.quant_proj(z)         # Shape: [B, 8, 16, 16]

        # Bottleneck
        q_low, vq_loss, perplexity = self.vq(z_low) # <--- FIXED: Pass z_low (8 channels) instead of z

        q = self.post_quant_proj(q_low)    # Shape: [B, 64, 16, 16]

        # Decoder
        h_q = self.from_bottleneck(q)
        out = self.decoder_backbone(h_q)

        return out, vq_loss, perplexity

import torch.nn as nn
import torch.nn.functional as F

class VectorQuantizer(nn.Module):
    def __init__(self, n_codes=256, dim=64, commit=0.25, decay=0.99, eps=1e-5,
             reset_every=200, reset_threshold=1.0):
        super().__init__()
        self.dim, self.commit, self.decay, self.eps, self.n_codes = dim, commit, decay, eps, n_codes

        # embeddings / codebook
        embed = torch.randn(n_codes, dim)

        self.register_buffer("embed", embed)
        self.register_buffer("cluster_size", torch.zeros(n_codes))
        self.register_buffer("embed_avg", embed.clone())

        # variables to reset dead codebook entries
        self.reset_every, self.reset_threshold = reset_every, reset_threshold
        self._step = 0

    @torch.no_grad()
    def _reset_dead_codes(self, flat):                    # flat: (BHW, D) encoder outputs
      dead = self.cluster_size < self.reset_threshold
      n_dead = int(dead.sum())
      if n_dead == 0:
          return

      # randomly select an embedding vector to replace one of the dead vectors
      pick = torch.randint(0, flat.size(0), (n_dead,), device=flat.device)
      new = flat[pick] + 1e-3 * torch.randn(n_dead, self.dim, device=flat.device)

      self.embed[dead] = new
      self.embed_avg[dead] = new
      self.cluster_size[dead] = 1.0


    def forward(self, z):                       # z: (B, D, H, W)
        B, D, H, W = z.shape
        flat = z.permute(0, 2, 3, 1).reshape(-1, D)

        # calculate distance from each feature vector and each codebook entry
        # but with a clever mathmatical trick :)
        d = flat.pow(2).sum(1, keepdim=True) - 2 * flat @ self.embed.t() + self.embed.pow(2).sum(1)
        idx = d.argmin(1)
        q = self.embed[idx].view(B, H, W, D).permute(0, 3, 1, 2)

        # we need onehot encoding to simplify EMA calculations
        onehot = F.one_hot(idx, self.n_codes).type(flat.dtype)

        if self.training:
              with torch.no_grad():
                # update codebook usage
                self.cluster_size.mul_(self.decay).add_(onehot.sum(0), alpha=1 - self.decay)

                #EMA: (B*EMA + (1-B)new embedding EMA) | updating codebook values
                self.embed_avg.mul_(self.decay).add_(onehot.t() @ flat, alpha=1 - self.decay)
                n = self.cluster_size.sum()
                cs = (self.cluster_size + self.eps) / (n + self.n_codes * self.eps) * n
                # move codebook entry to the average
                self.embed.copy_(self.embed_avg / cs.unsqueeze(1))

                self._step += 1
                if self.reset_every and (self._step % self.reset_every == 0):
                    self._reset_dead_codes(flat)

        commit_loss = F.mse_loss(z, q.detach())
        vq_loss = self.commit * commit_loss     # codebook is EMA-updated, so only the commit term remains
        q = z + (q - z).detach()                # straight-through

        # optional, to make it more verbose
        probs = onehot.mean(0)
        perplexity = torch.exp(-(probs * (probs + 1e-10).log()).sum())

        return q, vq_loss, perplexity

In [ ]:
USERNAME = "frost000"          # <-- fill in
REPO     = f"{USERNAME}/vqvae-anime-faces"

ckpt_path = hf_hub_download(REPO, "vqvae_checkpoint.pth")
ckpt = torch.load(ckpt_path, map_location=device)

arch = ckpt["arch"]          # {dim, n_codes, grid, img_size}
GRID = arch["grid"]          # generate() / encode_indices() need this global

vqvae = VQVAE(dim=arch["dim"], n_codes=arch["n_codes"]).to(device)
vqvae.load_state_dict(ckpt["model"])

In [ ]:
vqvae.eval()
for p in vqvae.parameters():
    p.requires_grad_(False)

In [ ]:
import itertools
with torch.no_grad():
    zs = torch.cat([encode(b.to(device)) for b, _ in itertools.islice(train_loader, 8)])
LATENT_SCALE = 1.0 / zs.std().item()
print("LATENT_SCALE", LATENT_SCALE, "| raw std", zs.std().item())

In [ ]:
#@title encode / decode helpers
@torch.no_grad()
def encode(x):
    with torch.autocast(device, enabled=False):
        x = x.float().to(next(vqvae.parameters()).device)
        h = vqvae.encoder_backbone(x)
        z = vqvae.to_bottleneck(h)
        z_low = vqvae.quant_proj(z)
        q_low, _, _ = vqvae.vq(z_low)
        return vqvae.post_quant_proj(q_low)

@torch.no_grad()
def decode(q):
    h_q = vqvae.from_bottleneck(q)
    return vqvae.decoder_backbone(h_q)           # [-1,1] via Tanh

In [ ]:
#@title recon + codebook stats
batch, _ = next(iter(val_loader))
batch = batch[:6].to(device)

with torch.no_grad():
    recon, vq_loss, perplexity = vqvae(batch)

mse = torch.nn.functional.mse_loss(recon, batch).item()
print(f"recon MSE {mse:.4f} | perplexity {perplexity.item():.1f} / {arch['n_codes']} codes")

def denorm(x): return (x.clamp(-1, 1) + 1) / 2
rows = []
for i in range(batch.shape[0]):
    rows += [denorm(batch[i]).cpu(), denorm(recon[i]).cpu()]
grid = vutils.make_grid(rows, nrow=2, padding=2)
plt.figure(figsize=(4, 11)); plt.axis("off")
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title("original  |  reconstruction\n(this is the quality ceiling)"); plt.show()

In [ ]:
#@title LDM initialization
#@title schedulers
noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",   # cosine — good default for faces
)

sampling_scheduler = DDIMScheduler.from_config(noise_scheduler.config)
sampling_scheduler.set_timesteps(50)

print("train steps:", noise_scheduler.config.num_train_timesteps,
      "| DDIM sample steps:", len(sampling_scheduler.timesteps))

#@title latent-space UNet
from diffusers import UNet2DModel

model = UNet2DModel(
    sample_size=16,
    in_channels=arch["dim"],      # 64
    out_channels=arch["dim"],     # 64
    layers_per_block=2,
    block_out_channels=(128, 256, 256),
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
    up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print(f"UNet params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

#@title training config
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

TOTAL_STEPS   = 20000     # bump to 8k–15k for good results
LR            = 2e-4
GRAD_CLIP     = 1.0

LOG_EVERY     = 50       # record train loss
VAL_EVERY     = 250      # record val loss
PREVIEW_EVERY = 500      # generate + show samples
FLUSH_EVERY   = 500      # clear cell output

optimizer    = AdamW(model.parameters(), lr=LR)
lr_scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_STEPS)
scaler       = torch.amp.GradScaler(device)

train_hist, val_hist = [], []   # each: (step, loss)
print(f"configured for {TOTAL_STEPS:,} steps")

In [ ]:
#@title diffusion loss (Min-SNR-γ weighted) — VQVAE latents
SNR_GAMMA = 5.0

def diffusion_loss(images):
    latents = encode(images).to(images.dtype)                                   # frozen VQVAE, no grad
    latents = latents * LATENT_SCALE                             # normalize latent std ~1
    noise   = torch.randn_like(latents)
    t = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                      (latents.shape[0],), device=device).long()
    noisy   = noise_scheduler.add_noise(latents, noise, t)
    pred    = model(noisy, t).sample

    mse = F.mse_loss(pred, noise, reduction="none").mean(dim=[1, 2, 3])

    acp = noise_scheduler.alphas_cumprod.to(device)[t]
    snr = acp / (1 - acp)
    weight = torch.clamp(snr, max=SNR_GAMMA) / snr

    return (weight * mse).mean()

In [ ]:
#@title sampler + grid helper
LATENT_CHANNELS = arch["dim"]   # 64
LATENT_SIZE     = 16
@torch.no_grad()
def sample(n=16, scheduler=sampling_scheduler):
    was_training = model.training
    model.eval()
    lat = torch.randn(n, LATENT_CHANNELS, LATENT_SIZE, LATENT_SIZE, device=device)
    for t in scheduler.timesteps:
        pred = model(lat, t).sample
        lat  = scheduler.step(pred, t, lat).prev_sample
    imgs = (decode(lat).clamp(-1, 1) + 1) / 2
    if was_training: model.train()
    return imgs.cpu()

def show_grid(imgs, title="", cols=8):
    n = imgs.shape[0]; rows = math.ceil(n / cols)
    plt.figure(figsize=(1.6 * cols, 1.6 * rows))
    grid = vutils.make_grid(imgs, nrow=cols, padding=2)
    plt.imshow(grid.permute(1, 2, 0).numpy()); plt.axis("off")
    if title: plt.title(title)
    plt.show()

In [ ]:
#@title train
from IPython.display import clear_output

def cycle(loader):
    while True:
        for b in loader: yield b
train_iter = cycle(train_loader)

model.train()
running = 0.0
pbar = tqdm(range(1, TOTAL_STEPS + 1), desc="training")

for step in pbar:
    images, _ = next(train_iter)

    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device, enabled=(device == "cuda")):
        images = images.to(device, non_blocking=True)
        loss = diffusion_loss(images)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer); scaler.update()
    lr_scheduler.step()

    running += loss.item()

    if step % LOG_EVERY == 0:
        avg = running / LOG_EVERY; running = 0.0
        train_hist.append((step, avg))
        pbar.set_postfix(loss=f"{avg:.4f}", lr=f"{lr_scheduler.get_last_lr()[0]:.1e}")

    if step % VAL_EVERY == 0:
        val_hist.append((step, validate()))

    if step % FLUSH_EVERY == 0:
        clear_output(wait=True)

    if step % PREVIEW_EVERY == 0:
        show_grid(sample(n=8), title=f"samples @ step {step}", cols=8)

print("training complete ✓")

In [ ]:
#@title loss curves
plt.figure(figsize=(9, 4))
if train_hist:
    ts, ls = zip(*train_hist); plt.plot(ts, ls, label="train", lw=1.5)
if val_hist:
    vs, vl = zip(*val_hist);   plt.plot(vs, vl, label="val", lw=1.5, marker="o", ms=4)
plt.xlabel("step"); plt.ylabel("MSE (ε-prediction)")
plt.title("latent diffusion training"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
#@title generate a grid of faces
show_grid(sample(n=16), title="generated anime faces (50-step DDIM)", cols=8)

In [ ]:
#@title higher-quality — more DDIM steps
steps = 111
hq = DDIMScheduler.from_config(noise_scheduler.config)
hq.set_timesteps(steps)
show_grid(sample(n=16, scheduler=hq), title=f"generated anime faces ({steps}-step DDIM)", cols=8)